# OceanWatch — Entrega 1 · Requisitos 3 y 4

**Requisito 3 — Preguntas de negocio (A-E)** respondidas con Spark sobre los CSV del Volume,
justificando las decisiones con el plan de ejecución.

**Requisito 4 — Almacenamiento óptimo para un propósito:** se materializa la tabla Delta
y se demuestra la mejora frente a CSV y Parquet con bytes, archivos leídos y efecto de
`OPTIMIZE`.

Prerrequisito: haber corrido `01_ingesta_perfilamiento`, que deja los 7 CSV en
`/Volumes/mine4213/proyecto/data/csv`.

## Imports y carga

In [ ]:
import os
import time

from pyspark.sql import Window
from pyspark.sql import functions as f

CATALOG = "mine4213"
SCHEMA = "proyecto"
VOLUME = "data"

BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CSV_DIR = f"{BASE}/csv"

AIS_SCHEMA = """
    MMSI string,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG float,
    COG float,
    Heading float,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType smallint,
    Status smallint,
    Length float,
    Width float,
    Draft float,
    Cargo string,
    TransceiverClass string
"""


def leer_csv():
    return (
        spark.read
        .option("header", True)
        .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
        .schema(AIS_SCHEMA)
        .csv(f"{CSV_DIR}/AIS_2023_06_*.csv")
    )


ais = (
    leer_csv()
    .withColumn("fecha", f.to_date("BaseDateTime"))
)

ais.printSchema()

# Requisito 3 — Preguntas de negocio

## Bases analíticas mínimas

In [ ]:
mmsi_valido = (
    f.col("MMSI").isNotNull()
    & f.trim(f.col("MMSI")).rlike(r"^[0-9]{9}$")
)

coordenada_valida = (
    f.col("LAT").between(-90, 90)
    & f.col("LON").between(-180, 180)
)

ais_buques = (
    ais
    .filter(mmsi_valido)
)

ais_espacial = (
    ais
    .filter(coordenada_valida)
)

## A. ¿Cuántos buques distintos transmitieron cada día?

Se compara el resultado exacto obtenido mediante `countDistinct` con
`approx_count_distinct`.

Para esta pregunta se consideran únicamente MMSI con formato válido de nueve
dígitos, ya que el perfilamiento identificó identificadores anómalos que no
deben interpretarse automáticamente como buques distintos.

### Conteo exacto

In [ ]:
exactos = (
    ais_buques
    .groupBy("fecha")
    .agg(
        f.countDistinct("MMSI").alias("buques_exactos")
    )
    .orderBy("fecha")
)

print("PLAN CONTEO EXACTO")
exactos.explain("formatted")

In [ ]:
inicio = time.perf_counter()

exact_rows = exactos.collect()

tiempo_exacto = time.perf_counter() - inicio

print(f"Tiempo conteo exacto: {tiempo_exacto:.2f} segundos")

### Conteo aproximado

In [ ]:
aproximados = (
    ais_buques
    .groupBy("fecha")
    .agg(
        f.approx_count_distinct("MMSI").alias("buques_aproximados")
    )
    .orderBy("fecha")
)

print("PLAN CONTEO APROXIMADO")
aproximados.explain("formatted")

In [ ]:
inicio = time.perf_counter()

approx_rows = aproximados.collect()

tiempo_aproximado = time.perf_counter() - inicio

print(f"Tiempo conteo aproximado: {tiempo_aproximado:.2f} segundos")

### Comparación

In [ ]:
exact_map = {
    row["fecha"]: row["buques_exactos"]
    for row in exact_rows
}

approx_map = {
    row["fecha"]: row["buques_aproximados"]
    for row in approx_rows
}

comparacion_data = []

for fecha in sorted(exact_map.keys()):

    exacto = exact_map[fecha]
    aproximado = approx_map[fecha]

    error_abs = abs(aproximado - exacto)

    error_pct = (
        100 * error_abs / exacto
        if exacto > 0
        else 0
    )

    comparacion_data.append(
        (
            fecha,
            exacto,
            aproximado,
            error_abs,
            float(error_pct)
        )
    )

comparacion_a = spark.createDataFrame(
    comparacion_data,
    [
        "fecha",
        "buques_exactos",
        "buques_aproximados",
        "error_absoluto",
        "error_porcentual"
    ]
)

display(comparacion_a)

display(
    comparacion_a
    .agg(
        f.round(
            f.avg("error_porcentual"),
            4
        ).alias("error_porcentual_medio"),

        f.round(
            f.max("error_porcentual"),
            4
        ).alias("error_porcentual_maximo")
    )
)

## B. ¿Qué tipos de buque generan más tráfico?

Se calcula el Top 10 de códigos de tipo de buque según el número de posiciones
AIS transmitidas durante la semana.

Los códigos se enriquecen utilizando el catálogo oficial de VesselType de
Marine Cadastre. [catalogo](https://coast.noaa.gov/data/marinecadastre/ais/VesselTypeCodes2018.pdf)

Para la velocidad media se excluye SOG=102.3 porque el perfilamiento determinó
que representa velocidad no disponible y no una velocidad real.

### Catálogo de tipos

In [ ]:
catalogo = []

def agregar(codigo, grupo, descripcion):
    catalogo.append(
        (codigo, grupo, descripcion)
    )


# 0
agregar(
    0,
    "Not Available",
    "Not available or no ship, default"
)

# 1-19
for c in range(1, 20):
    agregar(
        c,
        "Other",
        "Reserved for future use"
    )


# 20-29: WIG
wig = {
    20: ("Other", "Wing in ground (WIG), all ships of this type"),
    21: ("Tug Tow", "Wing in ground (WIG), hazardous category A"),
    22: ("Tug Tow", "Wing in ground (WIG), hazardous category B"),
    23: ("Other", "Wing in ground (WIG), hazardous category C"),
    24: ("Other", "Wing in ground (WIG), hazardous category D"),
    25: ("Other", "Wing in ground (WIG), reserved for future use"),
    26: ("Other", "Wing in ground (WIG), reserved for future use"),
    27: ("Other", "Wing in ground (WIG), reserved for future use"),
    28: ("Other", "Wing in ground (WIG), reserved for future use"),
    29: ("Other", "Wing in ground (WIG), reserved for future use"),
}

for codigo, (grupo, descripcion) in wig.items():
    agregar(codigo, grupo, descripcion)


# 30-59
tipos_especiales = {
    30: ("Fishing", "Fishing"),
    31: ("Tug Tow", "Towing"),
    32: ("Tug Tow", "Towing: length exceeds 200m or breadth exceeds 25m"),
    33: ("Other", "Dredging or underwater operations"),
    34: ("Other", "Diving operations"),
    35: ("Military", "Military operations"),
    36: ("Pleasure Craft/Sailing", "Sailing"),
    37: ("Pleasure Craft/Sailing", "Pleasure Craft"),
    38: ("Other", "Reserved"),
    39: ("Other", "Reserved"),

    40: ("Other", "High speed craft (HSC), all ships of this type"),
    41: ("Other", "High speed craft (HSC), hazardous category A"),
    42: ("Other", "High speed craft (HSC), hazardous category B"),
    43: ("Other", "High speed craft (HSC), hazardous category C"),
    44: ("Other", "High speed craft (HSC), hazardous category D"),
    45: ("Other", "High speed craft (HSC), reserved for future use"),
    46: ("Other", "High speed craft (HSC), reserved for future use"),
    47: ("Other", "High speed craft (HSC), reserved for future use"),
    48: ("Other", "High speed craft (HSC), reserved for future use"),
    49: ("Other", "High speed craft (HSC), no additional information"),

    50: ("Other", "Pilot Vessel"),
    51: ("Other", "Search and Rescue vessel"),
    52: ("Tug Tow", "Tug"),
    53: ("Other", "Port Tender"),
    54: ("Other", "Anti-pollution equipment"),
    55: ("Other", "Law Enforcement"),
    56: ("Other", "Spare - for assignment to local vessel"),
    57: ("Other", "Spare - for assignment to local vessel"),
    58: ("Other", "Medical Transport"),
    59: ("Other", "Ship according to RR Resolution No. 18"),
}

for codigo, (grupo, descripcion) in tipos_especiales.items():
    agregar(codigo, grupo, descripcion)

In [ ]:
familias = {
    60: ("Passenger", "Passenger"),
    70: ("Cargo", "Cargo"),
    80: ("Tanker", "Tanker"),
    90: ("Other", "Other Type")
}

sufijos = {
    0: "all ships of this type",
    1: "hazardous category A",
    2: "hazardous category B",
    3: "hazardous category C",
    4: "hazardous category D",
    5: "reserved for future use",
    6: "reserved for future use",
    7: "reserved for future use",
    8: "reserved for future use",
    9: "no additional information"
}

for base, (grupo, nombre) in familias.items():

    for offset in range(10):

        codigo = base + offset

        agregar(
            codigo,
            grupo,
            f"{nombre}, {sufijos[offset]}"
        )

In [ ]:
for c in range(100, 200):
    agregar(
        c,
        "Other",
        "Reserved for regional use"
    )

for c in range(200, 256):
    agregar(
        c,
        "Other",
        "Reserved for future use"
    )

for c in range(256, 1000):
    agregar(
        c,
        "Other",
        "No designation"
    )

In [ ]:
avis = {
    1001: ("Fishing", "Commercial Fishing Vessel"),
    1002: ("Fishing", "Fish Processing Vessel"),
    1003: ("Cargo", "Freight Barge"),
    1004: ("Cargo", "Freight Ship"),
    1005: ("Other", "Industrial Vessel"),
    1006: ("Other", "Miscellaneous Vessel"),
    1007: ("Other", "Mobile Offshore Drilling Unit"),
    1008: ("Other", "Non-vessel"),
    1009: ("Other", "NON-VESSEL"),
    1010: ("Other", "Offshore Supply Vessel"),
    1011: ("Other", "Oil Recovery"),
    1012: ("Passenger", "Passenger (Inspected)"),
    1013: ("Passenger", "Passenger (Uninspected)"),
    1014: ("Passenger", "Passenger Barge (Inspected)"),
    1015: ("Passenger", "Passenger Barge (Uninspected)"),
    1016: ("Cargo", "Public Freight"),
    1017: ("Tanker", "Public Tankship/Barge"),
    1018: ("Other", "Public Vessel, Unclassified"),
    1019: ("Pleasure Craft/Sailing", "Recreational"),
    1020: ("Other", "Research Vessel"),
    1021: ("Military", "SAR Aircraft"),
    1022: ("Other", "School Ship"),
    1023: ("Tug Tow", "Tank Barge"),
    1024: ("Tanker", "Tank Ship"),
    1025: ("Tug Tow", "Towing Vessel")
}

for codigo, (grupo, descripcion) in avis.items():
    agregar(
        codigo,
        grupo,
        descripcion
    )

In [ ]:
catalogo_tipos = spark.createDataFrame(
    catalogo,
    [
        "VesselType",
        "grupo_buque",
        "descripcion_tipo"
    ]
)

display(catalogo_tipos.limit(20))

### Respuesta

In [ ]:
ais_velocidad = (
    ais
    .withColumn(
        "SOG_utilizable",
        f.when(
            f.col("SOG").between(0, 102.2),
            f.col("SOG")
        )
    )
)

In [ ]:
top10_tipos_base = (
    ais_velocidad
    .filter(
        f.col("VesselType").isNotNull()
    )
    .groupBy("VesselType")
    .agg(
        f.count("*").alias("numero_posiciones"),

        f.round(
            f.avg("SOG_utilizable"),
            3
        ).alias("velocidad_media_nudos"),

        f.count("SOG_utilizable").alias(
            "posiciones_con_velocidad_utilizable"
        )
    )
    .orderBy(
        f.desc("numero_posiciones")
    )
    .limit(10)
)

In [ ]:
resultado_b = (
    top10_tipos_base
    .join(
        f.broadcast(catalogo_tipos),
        on="VesselType",
        how="left"
    )
    .select(
        "VesselType",
        "grupo_buque",
        "descripcion_tipo",
        "numero_posiciones",
        "velocidad_media_nudos",
        "posiciones_con_velocidad_utilizable"
    )
    .orderBy(
        f.desc("numero_posiciones")
    )
)

display(resultado_b)

In [ ]:
resultado_b.explain("formatted")

## C. ¿Qué 10 buques recorrieron más distancia durante la semana?

In [ ]:
window_vessel = Window.partitionBy("MMSI").orderBy("BaseDateTime")

df_lag = (ais
    .withColumn("LAT_prev", f.lag("LAT").over(window_vessel))
    .withColumn("LON_prev", f.lag("LON").over(window_vessel))
    .filter(f.col("LAT_prev").isNotNull() & f.col("LON_prev").isNotNull())
)

lat1 = f.radians(f.col("LAT_prev"))
lon1 = f.radians(f.col("LON_prev"))
lat2 = f.radians(f.col("LAT"))
lon2 = f.radians(f.col("LON"))

dlat = lat2 - lat1
dlon = lon2 - lon1

R_NM = 3440.0654

a = (f.sin(dlat / 2) ** 2) + f.cos(lat1) * f.cos(lat2) * (f.sin(dlon / 2) ** 2)
c = 2 * f.atan2(f.sqrt(a), f.sqrt(1 - a))
distancia_tramo = R_NM * c

df_distancias = (df_lag
    .withColumn("distancia_nm", distancia_tramo)
    .filter(f.col("distancia_nm") < 100)
)

top10_distancia = (df_distancias
    .groupBy("MMSI", "VesselName")
    .agg(
        f.round(f.sum("distancia_nm"), 2).alias("distancia_total_millas_nauticas"),
        f.round(f.try_divide(f.sum("distancia_nm"), f.avg("SOG")),2).alias("tiempo_total_horas")
    )
    .orderBy(f.col("distancia_total_millas_nauticas").desc())
    .limit(10)
)

display(top10_distancia)

In [ ]:
top10_distancia.explain("formatted")

## D. ¿Dónde se concentra el tráfico?

In [ ]:
df_h3 = ais.withColumn("h3_cell", f.expr("h3_longlatash3(LON, LAT, 8)"))

cells = (
    df_h3
    .groupBy("h3_cell")
    .agg(
        f.count("*").alias("num_posiciones"),
        f.round(f.avg("LAT"), 4).alias("lat_centroide"),
        f.round(f.avg("LON"), 4).alias("lon_centroide")
    )
    .orderBy(f.col("num_posiciones").desc())
    .limit(10)
)

display(cells)

In [ ]:
cells.explain("formatted")

Falta lo de puertos

## E. ¿Qué proporción de los buques de la semana transmitió los 7 días? ¿Dónde están los "visitantes de un solo día"?

### 1. Proporción de buques que transmitieron los 7 días

In [ ]:
df_dias_actividad = (ais
    .groupBy("MMSI")
    .agg(f.countDistinct("fecha").alias("dias_activos"))
)

proporcion_7_dias = (df_dias_actividad
    .select(
        f.count("MMSI").alias("total_buques"),
        f.sum(f.when(f.col("dias_activos") == 7, 1).otherwise(0)).alias("buques_7_dias"),
        f.sum(f.when(f.col("dias_activos") == 1, 1).otherwise(0)).alias("buques_1_dia")
    )
    .withColumn("porcentaje_7_dias", f.round((f.col("buques_7_dias") / f.col("total_buques")) * 100, 2))
    .withColumn("porcentaje_1_dia", f.round((f.col("buques_1_dia") / f.col("total_buques")) * 100, 2))
)

display(proporcion_7_dias)

La proporción de buques que vistaron los 7 días es: $$ \frac{12667}{31871} $$

Lo cual representa un 39.74% de todos los buques.

### 2. Ubicación de los visitantes de un solo día

In [ ]:
mmsi_visitantes_1_dia = df_dias_actividad.filter(f.col("dias_activos") == 1).select("MMSI")

df_visitantes = ais.join(mmsi_visitantes_1_dia, on="MMSI", how="inner")

display(df_visitantes.limit(100))

In [ ]:
ubicacion_visitantes = (df_visitantes
    .withColumn("h3_cell", f.expr("h3_longlatash3(LON, LAT, 8)"))
    .groupBy("h3_cell")
    .agg(
        f.count("*").alias("total_posiciones"),
        f.countDistinct("MMSI").alias("num_buques_visitantes"),
        f.round(f.avg("LAT"), 4).alias("lat_promedio"),
        f.round(f.avg("LON"), 4).alias("lon_promedio")
    )
    .orderBy(f.col("num_buques_visitantes").desc())
)

display(ubicacion_visitantes)

In [ ]:
ubicacion_visitantes.explain("formatted")

# Requisito 4 — Almacenamiento óptimo para un propósito

## 4.1 Propósito de consulta

**La consulta diaria del operador portuario:** para un día dado y una zona marítima
(bounding box lat/lon), cuántas posiciones y cuántos buques distintos hubo por hora.

Como ejemplo se usa la zona de Houston / Galveston (uno de los puertos con más tráfico
del corpus) el 3 de junio. El filtro combina **fecha** (igualdad) y **LAT/LON** (rangos),
así que el layout debe permitir descartar archivos por ambas dimensiones.

Se comparan cinco alternativas con la misma consulta:

| Alternativa | Formato | Layout |
|---|---|---|
| `csv` | CSV en el Volume | ninguno (1 archivo por día) |
| `parquet` | Parquet en el Volume | ninguno |
| `delta_base` | Delta | ninguno |
| `delta_particionada` | Delta | `PARTITIONED BY (fecha)` |
| `delta_cluster` | Delta | `CLUSTER BY (fecha, LAT, LON)` (liquid clustering) |

In [ ]:
FECHA_CONSULTA = "2023-06-03"

LAT_MIN, LAT_MAX = 29.0, 30.0
LON_MIN, LON_MAX = -95.5, -94.5

PARQUET_DIR = f"{BASE}/parquet/ais"

TABLA_BASE = f"{CATALOG}.{SCHEMA}.ais_delta_base"
TABLA_PARTICIONADA = f"{CATALOG}.{SCHEMA}.ais_delta_particionada"
TABLA_CLUSTER = f"{CATALOG}.{SCHEMA}.ais_posiciones"


def filtro_proposito(fecha_col):
    return (
        (fecha_col == f.lit(FECHA_CONSULTA).cast("date"))
        & f.col("LAT").between(LAT_MIN, LAT_MAX)
        & f.col("LON").between(LON_MIN, LON_MAX)
    )


def consulta_proposito(df, fecha_col):
    return (
        df
        .filter(filtro_proposito(fecha_col))
        .groupBy(f.hour("BaseDateTime").alias("hora"))
        .agg(
            f.count("*").alias("posiciones"),
            f.countDistinct("MMSI").alias("buques")
        )
        .orderBy("hora")
    )

## 4.2 Materialización de las alternativas

### Parquet sin layout

In [ ]:
inicio = time.perf_counter()

(
    ais
    .write
    .mode("overwrite")
    .parquet(PARQUET_DIR)
)

print(f"Parquet escrito en {time.perf_counter() - inicio:.2f} segundos")

### Delta sin layout

In [ ]:
inicio = time.perf_counter()

(
    ais
    .write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TABLA_BASE)
)

print(f"{TABLA_BASE} escrita en {time.perf_counter() - inicio:.2f} segundos")

### Delta particionada por fecha

7 particiones de ~8,6 M filas: la cardinalidad de `fecha` es baja, así que no genera
el problema de archivos pequeños. Particionar además por zona sí lo generaría (miles de
combinaciones fecha × celda con pocos datos cada una).

In [ ]:
inicio = time.perf_counter()

(
    ais
    .write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .partitionBy("fecha")
    .saveAsTable(TABLA_PARTICIONADA)
)

print(f"{TABLA_PARTICIONADA} escrita en {time.perf_counter() - inicio:.2f} segundos")

### Delta con liquid clustering por (fecha, LAT, LON)

`CLUSTER BY` ordena los datos con una curva de Hilbert sobre las tres columnas, de modo
que cada archivo cubre un rango acotado de fecha **y** de coordenadas. Delta guarda el
min/max de cada columna por archivo, y con eso descarta (data skipping) los archivos
cuyo rango no intersecta el filtro. Esta es la tabla final del proyecto.

In [ ]:
ais.createOrReplaceTempView("ais_csv")

inicio = time.perf_counter()

spark.sql(f"""
    CREATE OR REPLACE TABLE {TABLA_CLUSTER}
    CLUSTER BY (fecha, LAT, LON)
    COMMENT 'Posiciones AIS NOAA 1-7 junio 2023. Liquid clustering por (fecha, LAT, LON) para la consulta diaria del operador por fecha y zona marítima.'
    AS SELECT * FROM ais_csv
""")

print(f"{TABLA_CLUSTER} escrita en {time.perf_counter() - inicio:.2f} segundos")

## 4.3 Bytes en disco

In [ ]:
def bytes_directorio(path):
    total_bytes = 0
    total_archivos = 0

    for root, _, files in os.walk(path):
        for name in files:
            if name.endswith(".csv") or name.endswith(".parquet"):
                total_bytes += os.path.getsize(os.path.join(root, name))
                total_archivos += 1

    return total_archivos, total_bytes


def detalle_delta(tabla):
    detalle = spark.sql(f"DESCRIBE DETAIL {tabla}").first()
    return detalle["numFiles"], detalle["sizeInBytes"]


almacenamiento = [
    ("csv", *bytes_directorio(CSV_DIR)),
    ("parquet", *bytes_directorio(PARQUET_DIR)),
    ("delta_base", *detalle_delta(TABLA_BASE)),
    ("delta_particionada", *detalle_delta(TABLA_PARTICIONADA)),
    ("delta_cluster", *detalle_delta(TABLA_CLUSTER)),
]

almacenamiento_df = (
    spark.createDataFrame(
        almacenamiento,
        ["alternativa", "archivos", "bytes"]
    )
    .withColumn("GB", f.round(f.col("bytes") / 1024**3, 3))
    .withColumn(
        "porcentaje_vs_csv",
        f.round(
            100 * f.col("bytes") / f.lit(almacenamiento[0][2]),
            2
        )
    )
)

display(almacenamiento_df)

## 4.4 Archivos leídos por la consulta del propósito

Para cada alternativa se mide:

- **tiempo** de la consulta del propósito;
- **archivos con filas del filtro**: archivos distintos (`_metadata.file_path`, el
  equivalente a `INPUT_FILE_NAME`) que contienen alguna fila que cumple el filtro. Es el
  mínimo de archivos que un layout ideal tendría que abrir;
- **archivos totales** de la alternativa.

El conteo real de archivos leídos vs. podados (`files read` / `files pruned`) se ve en el
plan de abajo (`PartitionFilters`, `DataFilters`) y en el *Query Profile* de cada celda
(*See performance → Query profile*).

In [ ]:
fuentes = {
    "csv": (lambda: leer_csv(), f.to_date("BaseDateTime")),
    "parquet": (lambda: spark.read.parquet(PARQUET_DIR), f.col("fecha")),
    "delta_base": (lambda: spark.table(TABLA_BASE), f.col("fecha")),
    "delta_particionada": (lambda: spark.table(TABLA_PARTICIONADA), f.col("fecha")),
    "delta_cluster": (lambda: spark.table(TABLA_CLUSTER), f.col("fecha")),
}

archivos_totales = {
    nombre: archivos
    for nombre, archivos, _ in almacenamiento
}


def medir_consulta(nombre):
    cargar, fecha_col = fuentes[nombre]

    inicio = time.perf_counter()
    consulta_proposito(cargar(), fecha_col).collect()
    segundos = time.perf_counter() - inicio

    archivos_con_filas = (
        cargar()
        .filter(filtro_proposito(fecha_col))
        .select(f.col("_metadata.file_path"))
        .distinct()
        .count()
    )

    return (
        nombre,
        round(segundos, 2),
        archivos_con_filas,
        archivos_totales[nombre]
    )

In [ ]:
mediciones = [
    medir_consulta(nombre)
    for nombre in fuentes
]

mediciones_df = spark.createDataFrame(
    mediciones,
    [
        "alternativa",
        "segundos",
        "archivos_con_filas_del_filtro",
        "archivos_totales"
    ]
)

display(mediciones_df)

In [ ]:
display(consulta_proposito(spark.table(TABLA_CLUSTER), f.col("fecha")))

### Planes de ejecución

En CSV el filtro por fecha no se puede empujar (la fecha se deriva de `BaseDateTime` al
leer) y se escanean los 7 archivos completos. En la tabla particionada aparece
`PartitionFilters: fecha = 2023-06-03`; en la clusterizada, los filtros de fecha y
LAT/LON quedan como `DataFilters` que Delta usa para data skipping por estadísticas.

In [ ]:
for nombre in ["csv", "delta_particionada", "delta_cluster"]:
    cargar, fecha_col = fuentes[nombre]

    print("=" * 100)
    print(nombre)
    print("=" * 100)

    consulta_proposito(cargar(), fecha_col).explain("formatted")

## 4.5 Efecto de OPTIMIZE

En la tabla sin layout `OPTIMIZE` compacta archivos pequeños; en la tabla con
liquid clustering además (re)agrupa los datos por las claves de clustering.

Nota: en serverless las escrituras ya vienen con *optimized writes* y puede estar activa
la *predictive optimization*, así que el efecto puede ser menor que en un clúster clásico.
`DESCRIBE HISTORY` muestra qué hizo exactamente cada `OPTIMIZE`.

In [ ]:
antes_optimize = {
    tabla: detalle_delta(tabla)
    for tabla in [TABLA_BASE, TABLA_CLUSTER]
}

consulta_antes = medir_consulta("delta_cluster")

for tabla in [TABLA_BASE, TABLA_CLUSTER]:
    display(spark.sql(f"OPTIMIZE {tabla}"))

despues_optimize = {
    tabla: detalle_delta(tabla)
    for tabla in [TABLA_BASE, TABLA_CLUSTER]
}

consulta_despues = medir_consulta("delta_cluster")

In [ ]:
efecto_optimize = spark.createDataFrame(
    [
        (
            tabla,
            antes_optimize[tabla][0],
            despues_optimize[tabla][0],
            antes_optimize[tabla][1],
            despues_optimize[tabla][1]
        )
        for tabla in [TABLA_BASE, TABLA_CLUSTER]
    ],
    [
        "tabla",
        "archivos_antes",
        "archivos_despues",
        "bytes_antes",
        "bytes_despues"
    ]
)

display(efecto_optimize)

print("Consulta del propósito en delta_cluster (nombre, segundos, archivos con filas, archivos totales)")
print("Antes de OPTIMIZE:  ", consulta_antes)
print("Después de OPTIMIZE:", consulta_despues)

In [ ]:
display(
    spark.sql(f"DESCRIBE HISTORY {TABLA_CLUSTER}")
    .select("version", "timestamp", "operation", "operationMetrics")
)

## 4.6 Decisión

_Completar con los números obtenidos arriba._ Criterios:

- **Formato de archivo:** Parquet/Delta (columnar y comprimido) vs CSV: bytes en disco y
  columnas leídas (la consulta solo necesita `BaseDateTime`, `LAT`, `LON`, `MMSI`).
- **Formato de tabla:** Delta sobre Parquet agrega el log de transacciones con
  estadísticas min/max por archivo (data skipping), `OPTIMIZE`, clustering y time travel.
- **Layout:** partición por `fecha` poda por día pero no por zona; `CLUSTER BY (fecha, LAT, LON)`
  poda por ambas sin generar archivos pequeños y sin fijar un esquema de particiones
  difícil de cambiar.

Las tablas `ais_delta_base` y `ais_delta_particionada` y el directorio `parquet/` existen
solo como evidencia de la comparación; la tabla final es `mine4213.proyecto.ais_posiciones`.